In [0]:
dbutils.widgets.dropdown("env", "dev", ["dev", "uat", "prod"])

env = dbutils.widgets.get("env")

catalog = env
silver = "silver"
gold = "gold"

In [0]:
orders = spark.table(f"{catalog}.{silver}.silver_orders")
order_items = spark.table(f"{catalog}.{silver}.silver_order_items")
customers = spark.table(f"{catalog}.{silver}.silver_customers")
products = spark.table(f"{catalog}.{silver}.silver_products")

#### TASK 4: SPARK OPTIMIZATION (APPLIED DURING JOIN)

##### BROADCAST JOIN

In [0]:
from pyspark.sql.functions import broadcast

fact_df = orders.alias("o") \
    .join(order_items.alias("oi"), "order_id") \
    .join(broadcast(customers.alias("c")), "customer_id") \
    .join(broadcast(products.alias("p")), "product_id")

# Why?
# customers, products → small tables
# avoids shuffle → faster joins

In [0]:
from pyspark.sql.functions import col

fact_df = fact_df.select(
    "o.order_id",
    "o.customer_id",
    "oi.product_id",
    "o.order_date",
    "c.state",
    "p.category",
    "oi.quantity",
    col("oi.price").alias("sale_price")
)

#### REPARTION (BETTER PARALLELISM)

In [0]:
fact_df = fact_df.repartition(10)


# Why?
# Distributes data evenly
# Improves write performance

#### CACHE
- Use only if multiple aggregations below

In [0]:

#### PERSIST TABLE is not supported on serverless compute. SQLSTATE: 0A000

### fact_df.cache()



##### TASK 3: DATA MODELING (FACT + DIM)

####### Partitioning Strategy
- Partition by state → common filter column
- Enables partition pruning

In [0]:
fact_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("state") \
    .saveAsTable(f"{catalog}.{gold}.fact_sales")

In [0]:
customers.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold}.dim_customers")

In [0]:
products.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold}.dim_products")

#### AGGREGATIONS

In [0]:
from pyspark.sql.functions import sum

revenue_state = fact_df.groupBy("state") \
    .agg(sum("sale_price").alias("total_revenue"))

In [0]:
top_products = fact_df.groupBy("product_id") \
    .agg(sum("sale_price").alias("revenue")) \
    .orderBy(col("revenue").desc())

In [0]:
from pyspark.sql.functions import to_date

daily_sales = fact_df.groupBy(to_date("order_date").alias("date")) \
    .agg(sum("sale_price").alias("daily_revenue"))

In [0]:
from pyspark.sql.functions import month

monthly_sales = fact_df.groupBy(month("order_date").alias("month")) \
    .agg(sum("sale_price").alias("monthly_revenue"))

#### TASK 4: COALESCE (FILE OPTIMIZATION)

In [0]:
fact_df = fact_df.coalesce(5)

# Why?
# Reduce small files problem
# No shuffle → efficient

#### TASK 4: SKEW HANDLING (ADVANCED)

In [0]:
fact_df = orders.hint("skew") \
    .join(order_items, "order_id")

#We can also use bradcast, its already done.

##### TASK 5: DELTA LAKE OPTIMIZATION

##### OPTIMIZE + ZORDER

In [0]:
spark.sql(f"""
OPTIMIZE {catalog}.gold.fact_sales
ZORDER BY (customer_id, product_id)
""")

# Why?
# Faster filtering on customer/product

##### VACUUM

In [0]:
spark.sql(f"""
VACUUM {catalog}.gold.fact_sales RETAIN 168 HOURS
""")

# Why ? - Removes old unused files


#### TIME TRAVEL

In [0]:
spark.sql(f"""
SELECT * FROM {catalog}.gold.fact_sales VERSION AS OF 1
""")

# Why - Access the previous versions of the object

##### SCHEMA ENFORCEMENT
- Already applied implicitly via Delta

- Optional strict enforcement:

In [0]:
fact_df.write.format("delta") \
    .option("overwriteSchema", "true")

#### PARTITION PRUNING

####### Happens automatically when querying:
####### Only that partition is read 


In [0]:
spark.sql(f"SELECT * FROM {catalog}.gold.fact_sales WHERE state = 'MH'")